# Parkinson's Disease Detection – GPU Colab Notebook

This notebook downloads the Parkinson's telemonitoring dataset, preprocesses it, builds a tiny VGG‑style model for tabular data, trains it on a GPU, and saves the trained model for download.


In [ ]:
# 1️⃣ Install required packages
!pip -q install tensorflow pandas scikit-learn matplotlib seaborn

## 2️⃣ Load the dataset
We use the UCI Parkinson's telemonitoring CSV. If you already have a local copy, you can upload it via the Colab file‑upload widget instead.


In [ ]:
import pandas as pd, numpy as np
# Option 1: download directly
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/telemonitoring/parkinsons_updrs.csv'
df = pd.read_csv(url)
# Option 2: if you upload a file named 'parkinsons_updrs.csv' in the Colab session, replace df with the uploaded file:
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv('parkinsons_updrs.csv')
df.head()

## 3️⃣ Pre‑process the data
* Drop identifier columns (`subject#`, `test_time`).
* Create a binary label: `total_UPDRS > 15` (1 = Parkinson's, 0 = healthy).
* Scale numeric features.
* Train/validation split (80/20).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create binary label
df['label'] = (df['total_UPDRS'] > 15).astype(int)
# Features – drop identifiers and the target/label
X = df.drop(columns=['subject#', 'test_time', 'total_UPDRS', 'label']).values.astype(float)
y = df['label'].values

# Train / validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
print('Train shape:', X_train.shape, 'Val shape:', X_val.shape)

## 4️⃣ Build a tiny VGG‑like model for tabular data
We reshape the 22‑dimensional feature vector into a (2, 11, 1) “image” so that Conv2D layers can be used.


In [ ]:
import tensorflow as tf

@tf.keras.utils.register_keras_serializable()
def reshape_features(x):
    # x: (batch, 22) -> (batch, 2, 11, 1)
    x = tf.expand_dims(x, -1)               # (batch, 22, 1)
    x = tf.reshape(x, (-1, 2, 11, 1))       # (batch, 2, 11, 1)
    return x

inputs = tf.keras.Input(shape=(X_train.shape[1],), name='features')
x = tf.keras.layers.Lambda(reshape_features)(inputs)
# Tiny VGG‑like block
x = tf.keras.layers.Conv2D(32, (3,3), padding='same')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPooling2D((2,2))(x)
x = tf.keras.layers.Conv2D(64, (3,3), padding='same')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                          loss='binary_crossentropy',
                          metrics=['accuracy'])
model.summary()

## 5️⃣ Train the model (GPU)
The default Colab runtime will use a GPU if you set it in *Runtime → Change runtime type*.


In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    verbose=2
)
# Plot training curves
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.show()

## 6️⃣ Save & download the trained model
The model is saved as `best_trained.keras` and a download button is provided.


In [ ]:
model.save('best_trained.keras')
from google.colab import files
files.download('best_trained.keras')